# Exploratory Real-Data Analysis

This notebook develops a common exploratory analysis for three substantively different applications: Greater Glasgow respiratory hospitalizations, California lung-cancer incidence, and South Korean tracheal, bronchial, and lung cancer mortality (ICD-10 C33-C34). Each application is harmonized as areal counts, expected counts, a boundary-driving covariate, and a geographic adjacency graph. The analysis motivates a shared statistical model and reusable posterior approximator while allowing application-specific spatial and boundary regimes.

In [ ]:
from pathlib import Path
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize, TwoSlopeNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable

warnings.filterwarnings("ignore", category=UserWarning)

CWD = Path.cwd()
if (CWD / "Data" / "respiratory_data_glasgow.gpkg").exists():
    REAL_DATA_DIR = CWD
elif (CWD / "Real Data Analysis" / "Data" / "respiratory_data_glasgow.gpkg").exists():
    REAL_DATA_DIR = CWD / "Real Data Analysis"
else:
    raise FileNotFoundError("Could not locate the Real Data Analysis/Data folder from the current working directory.")

DATA_DIR = REAL_DATA_DIR / "Data"
OUTPUT_DIR = REAL_DATA_DIR / "results_exploratory_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 120,
})

print(f"Reading data from: {DATA_DIR}")
print(f"Saving figures to: {OUTPUT_DIR}")

In [ ]:
glasgow = gpd.read_file(DATA_DIR / "respiratory_data_glasgow.gpkg")
california = gpd.read_file(DATA_DIR / "respiratory_data_california.gpkg")
south_korea = gpd.read_file(DATA_DIR / "South_Korea" / "mortality_data_south_korea.gpkg")

required_columns = {
    "glasgow": (glasgow, ["SMR", "incomedep"]),
    "california": (california, ["lung_standard_ratio", "smoking"]),
    "south_korea": (south_korea, ["smr_lung_cancer", "smoking_pct"]),
}
for dataset_name, (gdf, columns) in required_columns.items():
    missing = [column for column in columns if column not in gdf.columns]
    if missing:
        raise KeyError(f"{dataset_name} is missing required columns: {missing}")

summary_rows = []
for dataset_name, gdf, columns in [
    ("Glasgow", glasgow, ["SMR", "incomedep"]),
    ("California", california, ["lung_standard_ratio", "smoking"]),
    ("South Korea", south_korea, ["smr_lung_cancer", "smoking_pct"]),
]:
    for column in columns:
        values = np.asarray(gdf[column], dtype=float)
        summary_rows.append({
            "dataset": dataset_name,
            "variable": column,
            "min": np.nanmin(values),
            "median": np.nanmedian(values),
            "max": np.nanmax(values),
            "missing": int(np.isnan(values).sum()),
        })

import pandas as pd
summary_df = pd.DataFrame(summary_rows)
display(summary_df)

In [ ]:
def get_plot_gdf(gdf):
    """Project geographic layers before plotting so map geometry is not visually distorted."""
    if gdf.crs is None or not getattr(gdf.crs, "is_geographic", False):
        return gdf

    try:
        plot_crs = gdf.estimate_utm_crs()
        if plot_crs is None:
            return gdf
        return gdf.to_crs(plot_crs)
    except Exception:
        return gdf


def add_choropleth(ax, gdf, column, title, cmap, colorbar_label, vmin=None, vmax=None):
    gdf_plot = get_plot_gdf(gdf).copy()
    values = np.asarray(gdf_plot[column], dtype=float)
    if vmin is None:
        vmin = float(np.nanmin(values))
    if vmax is None:
        vmax = float(np.nanmax(values))

    norm = Normalize(vmin=vmin, vmax=vmax)
    gdf_plot.plot(
        column=column,
        ax=ax,
        cmap=cmap,
        norm=norm,
        edgecolor="0.55",
        linewidth=0.35,
        legend=False,
    )
    ax.set_title(title, loc="left", pad=6)
    ax.set_axis_off()
    ax.set_aspect("equal")

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="3%", pad=0.04)
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = ax.figure.colorbar(sm, cax=cax)
    cbar.set_label(colorbar_label, fontsize=10)
    cbar.ax.tick_params(labelsize=9)
    return ax


def save_figure(fig, filename):
    path = OUTPUT_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight", pad_inches=0.03)
    print(f"Saved: {path}")

def style_boxed_axes(ax):
    ax.set_facecolor("white")
    ax.set_axisbelow(True)
    ax.grid(True, color="#b0b0b0", linewidth=0.8, alpha=0.35)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#262626")
        spine.set_linewidth(0.8)


In [ ]:
risk_limits = {
    "glasgow": (float(glasgow["SMR"].min()), float(glasgow["SMR"].max())),
    "california": (float(california["lung_standard_ratio"].min()), float(california["lung_standard_ratio"].max())),
    "south_korea": (float(south_korea["smr_lung_cancer"].min()), float(south_korea["smr_lung_cancer"].max())),
}


In [ ]:
# Figure 1 standalone maps are generated in the next cell. That cell applies
# one common map and legend geometry to all six displayed panels.


In [ ]:
# Rebuild Fig. 1 using the same map/legend geometry used for Fig. 3.
# Each exploratory panel is first saved as a standalone map, then rescaled to
# the corresponding boundary-agreement reference canvas before assembling the 2x2 figure.

from PIL import Image

MAP_FIGSIZE = (12, 12)
MAP_LEGEND_SHRINK = 0.75
MAP_LEGEND_TICKSIZE = 18
MAP_LEGEND_LABELSIZE = 18
FIG3_REFERENCE_DIR = REAL_DATA_DIR / "results_ABI_vs_CARBayes"


def style_map_legend(fig):
    if len(fig.axes) < 2:
        return
    cax = fig.axes[-1]
    cax.tick_params(labelsize=MAP_LEGEND_TICKSIZE)
    cax.yaxis.label.set_size(MAP_LEGEND_LABELSIZE)


def plot_single_map_like_fig3(gdf, column, cmap, colorbar_label, filename, vmin=None, vmax=None, center=None):
    gdf_plot = get_plot_gdf(gdf).copy()
    values = np.asarray(gdf_plot[column], dtype=float)
    if vmin is None:
        vmin = float(np.nanmin(values))
    if vmax is None:
        vmax = float(np.nanmax(values))
    norm = (
        TwoSlopeNorm(vmin=vmin, vcenter=center, vmax=vmax)
        if center is not None
        else Normalize(vmin=vmin, vmax=vmax)
    )

    fig, ax = plt.subplots(1, 1, figsize=MAP_FIGSIZE)
    gdf_plot.plot(
        column=column,
        ax=ax,
        cmap=cmap,
        edgecolor="0.55",
        linewidth=0.35,
        norm=norm,
        legend=True,
        legend_kwds={"label": colorbar_label, "shrink": MAP_LEGEND_SHRINK},
    )
    style_map_legend(fig)
    ax.axis("off")
    save_figure(fig, filename)
    plt.close(fig)


def _content_bbox(image, tolerance=8):
    arr = np.array(image.convert("RGBA"))
    bg = arr[0, 0].astype(int)
    diff = np.max(np.abs(arr.astype(int) - bg), axis=2)
    mask = diff > tolerance
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return (0, 0, image.width, image.height)
    return (int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1)


def _column_clusters(mask, min_pixels=10):
    counts = mask.sum(axis=0)
    cols = np.where(counts > min_pixels)[0]
    if len(cols) == 0:
        return []
    gaps = np.where(np.diff(cols) > 1)[0]
    starts = [cols[0]] + [cols[i + 1] for i in gaps]
    ends = [cols[i] for i in gaps] + [cols[-1]]
    return [(int(start), int(end) + 1) for start, end in zip(starts, ends)]


def _split_map_and_legend(image, tolerance=8, merge_gap=60):
    arr = np.array(image.convert("RGBA"))
    bg = arr[0, 0].astype(int)
    diff = np.max(np.abs(arr.astype(int) - bg), axis=2)
    mask = diff > tolerance
    clusters = _column_clusters(mask)
    if len(clusters) < 2:
        return image.crop(_content_bbox(image)), None

    legend_start, _ = clusters[-1]
    idx = len(clusters) - 2
    while idx >= 0 and legend_start - clusters[idx][1] <= merge_gap:
        legend_start = clusters[idx][0]
        idx -= 1

    if idx < 0:
        return image.crop(_content_bbox(image)), None

    map_part = image.crop((0, 0, legend_start, image.height))
    legend_part = image.crop((legend_start, 0, image.width, image.height))
    return map_part.crop(_content_bbox(map_part)), legend_part.crop(_content_bbox(legend_part))


def _resize_to_height(image, target_height):
    if image is None or image.height == target_height:
        return image
    new_width = int(round(image.width * target_height / image.height))
    return image.resize((new_width, target_height), Image.Resampling.LANCZOS)


def _match_target_to_reference_map_scale(reference_path, target_path, overwrite=True, gap_px=120, horizontal_padding_px=80):
    with Image.open(reference_path) as reference_raw, Image.open(target_path) as target_raw:
        reference_img = reference_raw.convert("RGBA")
        target_img = target_raw.convert("RGBA")

        reference_map, reference_legend = _split_map_and_legend(reference_img)
        target_map, target_legend = _split_map_and_legend(target_img)

        canvas_height = reference_img.height
        target_map = _resize_to_height(target_map, reference_map.height)
        if target_legend is not None:
            target_legend = _resize_to_height(
                target_legend,
                int(round(MAP_LEGEND_SHRINK * reference_map.height)),
            )

        background = tuple(int(x) for x in np.array(reference_img)[0, 0])
        group_width = target_map.width + (gap_px if target_legend is not None else 0) + (target_legend.width if target_legend is not None else 0)
        canvas_width = group_width + 2 * horizontal_padding_px
        canvas = Image.new("RGBA", (canvas_width, canvas_height), background)
        x0 = horizontal_padding_px
        map_y = (canvas_height - target_map.height) // 2
        canvas.paste(target_map, (x0, map_y), target_map)
        if target_legend is not None:
            legend_x = x0 + target_map.width + gap_px
            legend_y = (canvas_height - target_legend.height) // 2
            canvas.paste(target_legend, (legend_x, legend_y), target_legend)

        out_path = target_path if overwrite else target_path.with_stem(target_path.stem + "_same_scale")
        canvas.save(out_path)

        final_map, final_legend = _split_map_and_legend(canvas)
        print(
            f"Matched {target_path.name} to {reference_path.name}: "
            f"canvas={canvas.size}, map={final_map.size}, "
            f"legend={final_legend.size if final_legend is not None else None}"
        )


fig1_panels = [
    (glasgow, "SMR", "RdBu_r", "SHR", "glasgow_shr_map.png", *risk_limits["glasgow"], 1.0, FIG3_REFERENCE_DIR / "glasgow" / "glasgow_boundary_agreement.png"),
    (glasgow, "incomedep", "YlGn", "Income deprivation (%)", "glasgow_income_deprivation_map.png", None, None, None, FIG3_REFERENCE_DIR / "glasgow" / "glasgow_boundary_agreement.png"),
    (california, "lung_standard_ratio", "RdBu_r", "SIR", "california_lung_sir_map.png", *risk_limits["california"], 1.0, FIG3_REFERENCE_DIR / "california" / "california_boundary_agreement.png"),
    (california, "smoking", "YlGn", "Smoking (%)", "california_smoking_prevalence_map.png", None, None, None, FIG3_REFERENCE_DIR / "california" / "california_boundary_agreement.png"),
    (south_korea, "smr_lung_cancer", "RdBu_r", "SMR", "south_korea_lung_smr_map.png", *risk_limits["south_korea"], 1.0, FIG3_REFERENCE_DIR / "south_korea" / "south_korea_boundary_agreement.png"),
    (south_korea, "smoking_pct", "YlGn", "Smoking (%)", "south_korea_smoking_prevalence_map.png", None, None, None, FIG3_REFERENCE_DIR / "south_korea" / "south_korea_boundary_agreement.png"),
]

for gdf, column, cmap, label, filename, vmin, vmax, center, reference_path in fig1_panels:
    target_path = OUTPUT_DIR / filename
    if not reference_path.exists():
        raise FileNotFoundError(f"Missing Fig. 3 reference map: {reference_path}")
    plot_single_map_like_fig3(gdf, column, cmap, label, filename, vmin=vmin, vmax=vmax, center=center)
    _match_target_to_reference_map_scale(reference_path, target_path, overwrite=True)


## A common statistical template across different health applications

The three applications concern different outcomes, periods, spatial units, and public-health settings. They nevertheless share the observable structure required by the boundary model: areal counts $y_i$, expected counts $e_i$, a scientifically meaningful boundary-driving covariate $x_i$, and an adjacency graph $A$. The following cells harmonize those inputs and examine whether each application exhibits spatial organization, neighboring-area similarity, and larger local risk contrasts along covariate-dissimilar borders. These are descriptive diagnostics rather than causal analyses.

In [ ]:
from matplotlib.patches import Patch, Rectangle
from scipy.stats import spearmanr
from shapely.geometry import box
from shapely.ops import unary_union
import textwrap

APP_COLORS = {
    "glasgow": "#B55232",
    "california": "#2F6F8F",
    "south_korea": "#3D7A57",
}

APPLICATION_CONFIG = {
    "glasgow": dict(
        label="Greater Glasgow", prefix="G", gdf=glasgow, id_col="IZ", name_col="name",
        observed_col="observed", expected_col="expected", ratio_col="SMR", covariate_col="incomedep",
        adjacency_path=DATA_DIR / "adjacency_matrix_glasgow.csv",
        outcome="Respiratory hospitalizations", period="2010", spatial_unit="Intermediate Zone",
        ratio_label="SHR", covariate_label="Income deprivation (%)",
    ),
    "california": dict(
        label="California", prefix="C", gdf=california, id_col="county", name_col="county",
        observed_col="lung_O_count", expected_col="lung_E_count", ratio_col="lung_standard_ratio", covariate_col="smoking",
        adjacency_path=DATA_DIR / "adjacency_matrix_california.csv",
        outcome="Lung-cancer incidence", period="Study period", spatial_unit="County",
        ratio_label="SIR", covariate_label="Adult smoking prevalence (%)",
    ),
    "south_korea": dict(
        label="South Korea", prefix="K", gdf=south_korea, id_col="area_id", name_col="area_name",
        observed_col="observed_lung_cancer", expected_col="expected_lung_cancer", ratio_col="smr_lung_cancer", covariate_col="smoking_pct",
        adjacency_path=DATA_DIR / "South_Korea" / "adjacency_matrix_south_korea.csv",
        outcome="Tracheal, bronchial, and lung cancer mortality", period="2015-2019", spatial_unit="Municipality",
        ratio_label="SMR", covariate_label="Current smoking prevalence (%)",
    ),
}


def canonical_area_id(value):
    if pd.isna(value):
        return ""
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)) and float(value).is_integer():
        return str(int(value))
    return str(value).strip().lower()


def moran_i(values, adjacency):
    values = np.asarray(values, dtype=float)
    centered = values - np.nanmean(values)
    denominator = float(centered @ centered)
    weight_sum = float(adjacency.sum())
    if denominator <= 0 or weight_sum <= 0:
        return np.nan
    return len(values) / weight_sum * float(centered @ adjacency @ centered) / denominator


def harmonize_application(key, config):
    adjacency_df = pd.read_csv(config["adjacency_path"])
    adjacency = adjacency_df.to_numpy(dtype=np.int8)
    adjacency_ids = [canonical_area_id(column) for column in adjacency_df.columns]
    if adjacency.shape != (len(adjacency_ids), len(adjacency_ids)):
        raise ValueError(f"{key}: adjacency matrix is not square")
    if not np.array_equal(adjacency, adjacency.T) or np.any(np.diag(adjacency) != 0):
        raise ValueError(f"{key}: adjacency matrix must be symmetric with a zero diagonal")

    gdf = config["gdf"].copy()
    gdf["_area_key"] = gdf[config["id_col"]].map(canonical_area_id)
    if gdf["_area_key"].duplicated().any():
        raise ValueError(f"{key}: duplicated geographic identifiers")
    lookup = {area_key: idx for idx, area_key in enumerate(gdf["_area_key"])}
    missing = [area_key for area_key in adjacency_ids if area_key not in lookup]
    if missing:
        raise KeyError(f"{key}: adjacency IDs missing from geographic data: {missing[:5]}")
    gdf = gdf.iloc[[lookup[area_key] for area_key in adjacency_ids]].reset_index(drop=True)

    observed = gdf[config["observed_col"]].to_numpy(dtype=float)
    expected = gdf[config["expected_col"]].to_numpy(dtype=float)
    covariate = gdf[config["covariate_col"]].to_numpy(dtype=float)
    ratio = gdf[config["ratio_col"]].to_numpy(dtype=float)
    crude_log_risk = np.log(observed + 0.5) - np.log(expected)
    covariate_z = (covariate - covariate.mean()) / covariate.std(ddof=1)
    degree = adjacency.sum(axis=1).astype(float)
    edge_i, edge_j = np.where(np.triu(adjacency, 1) == 1)

    edge_df = pd.DataFrame({
        "from_index": edge_i,
        "to_index": edge_j,
        "from_id": [adjacency_ids[i] for i in edge_i],
        "to_id": [adjacency_ids[j] for j in edge_j],
        "from_name": gdf.iloc[edge_i][config["name_col"]].astype(str).to_numpy(),
        "to_name": gdf.iloc[edge_j][config["name_col"]].astype(str).to_numpy(),
        "edge_dissimilarity": np.abs(covariate_z[edge_i] - covariate_z[edge_j]),
        "risk_contrast": np.abs(crude_log_risk[edge_i] - crude_log_risk[edge_j]),
    })
    edge_df["dissimilarity_quintile"] = pd.qcut(
        edge_df["edge_dissimilarity"].rank(method="first"), 5, labels=["Q1", "Q2", "Q3", "Q4", "Q5"]
    )

    non_i, non_j = np.where(np.triu(1 - adjacency - np.eye(len(gdf), dtype=np.int8), 1) == 1)
    rng = np.random.default_rng(20260831 + len(gdf))
    sample_idx = rng.choice(len(non_i), size=min(len(edge_i), len(non_i)), replace=False)
    nonneighbor_contrast = np.abs(crude_log_risk[non_i[sample_idx]] - crude_log_risk[non_j[sample_idx]])

    display_width = 3
    gdf["display_id"] = [f"{config['prefix']}{idx:0{display_width}d}" for idx in range(1, len(gdf) + 1)]
    if key == "california":
        gdf["display_name"] = gdf[config["name_col"]].astype(str).str.title()
    else:
        gdf["display_name"] = gdf[config["name_col"]].astype(str)

    harmonized = pd.DataFrame({
        "display_id": gdf["display_id"],
        "area_id": adjacency_ids,
        "area_name": gdf["display_name"],
        "observed": observed,
        "expected": expected,
        "risk_ratio": ratio,
        "covariate": covariate,
        "crude_log_risk": crude_log_risk,
        "degree": degree,
    })
    harmonized.to_csv(OUTPUT_DIR / f"{key}_harmonized_data.csv", index=False)
    edge_df.to_csv(OUTPUT_DIR / f"{key}_raw_edge_diagnostics.csv", index=False)

    return dict(
        key=key, config=config, gdf=gdf, adjacency=adjacency, adjacency_ids=adjacency_ids,
        observed=observed, expected=expected, ratio=ratio, covariate=covariate,
        crude_log_risk=crude_log_risk, covariate_z=covariate_z, degree=degree,
        edge_df=edge_df, adjacent_contrast=edge_df["risk_contrast"].to_numpy(),
        nonneighbor_contrast=nonneighbor_contrast, harmonized=harmonized,
    )


applications = {key: harmonize_application(key, config) for key, config in APPLICATION_CONFIG.items()}
print("Harmonized applications:", {key: len(app["gdf"]) for key, app in applications.items()})

## Descriptive evidence for a shared model class

A shared posterior approximator does not require common epidemiological dynamics or shared parameters. It requires repeated posterior inference for the same statistical object. The diagnostics below show how the three applications share an offset-based count formulation, spatially organized crude risk, neighboring-area similarity, and a descriptive tendency toward larger risk contrasts at highly covariate-dissimilar borders, while retaining clearly different strengths and scales.

In [ ]:
def quantile_text(values):
    q1, median, q3 = np.quantile(np.asarray(values, dtype=float), [0.25, 0.5, 0.75])
    return f"{median:.3g} [{q1:.3g}, {q3:.3g}]"


summary_rows = []
schema_rows = []
for key, app in applications.items():
    cfg = app["config"]
    edge_grouped = app["edge_df"].groupby("dissimilarity_quintile", observed=True)["risk_contrast"].mean()
    low_high_ratio = float(edge_grouped.loc["Q5"] / max(edge_grouped.loc["Q1"], 1e-12))
    summary_rows.append({
        "Application": cfg["label"],
        "Areas": len(app["gdf"]),
        "Edges": len(app["edge_df"]),
        "Mean degree": app["degree"].mean(),
        "Observed counts, median [IQR]": quantile_text(app["observed"]),
        "Expected counts, median [IQR]": quantile_text(app["expected"]),
        "Risk ratio": cfg["ratio_label"],
        "Standardized risk, median [IQR]": quantile_text(app["ratio"]),
        "Covariate Moran I": moran_i(app["covariate"], app["adjacency"]),
        "Crude log-risk Moran I": moran_i(app["crude_log_risk"], app["adjacency"]),
        "Adjacent contrast": app["adjacent_contrast"].mean(),
        "Sampled non-neighbor contrast": app["nonneighbor_contrast"].mean(),
        "Q5/Q1 edge-contrast ratio": low_high_ratio,
        "Covariate-risk Spearman": spearmanr(app["covariate"], app["crude_log_risk"]).statistic,
    })
    schema_rows.append({
        "Application": cfg["label"], "Outcome": cfg["outcome"], "Period": cfg["period"],
        "Spatial unit": cfg["spatial_unit"], "Count input": cfg["observed_col"],
        "Expected-count input": cfg["expected_col"], "Boundary-driving covariate": cfg["covariate_label"],
    })

application_summary_df = pd.DataFrame(summary_rows)
application_schema_df = pd.DataFrame(schema_rows)
application_summary_df.to_csv(OUTPUT_DIR / "application_characteristics.csv", index=False)
application_schema_df.to_csv(OUTPUT_DIR / "application_common_input_schema.csv", index=False)
display(application_schema_df)
display(application_summary_df.round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), constrained_layout=True)
for ax, (key, app) in zip(axes, applications.items()):
    cfg = app["config"]
    color = APP_COLORS[key]
    ax.scatter(app["expected"], app["observed"], s=20, alpha=0.65, color=color, edgecolor="white", linewidth=0.3)
    lower = min(app["expected"].min(), app["observed"].min())
    upper = max(app["expected"].max(), app["observed"].max())
    ax.plot([lower, upper], [lower, upper], color="0.25", linestyle="--", linewidth=1)
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_title(cfg["label"], fontweight="bold")
    ax.set_xlabel("Expected count"); ax.set_ylabel("Observed count")
    style_boxed_axes(ax)
save_figure(fig, "application_observed_expected_relationships.png")
plt.show()

fig, axes = plt.subplots(2, 3, figsize=(15, 7.4), constrained_layout=True)
for column, (key, app) in enumerate(applications.items()):
    cfg = app["config"]
    color = APP_COLORS[key]

    ax = axes[0, column]
    violin = ax.violinplot([app["adjacent_contrast"], app["nonneighbor_contrast"]], positions=[1, 2], showmeans=True, showextrema=False)
    for body in violin["bodies"]:
        body.set_facecolor(color); body.set_edgecolor("white"); body.set_alpha(0.75)
    violin["cmeans"].set_color("black")
    ax.set_xticks([1, 2], ["Adjacent", "Sampled\nnon-neighbor"])
    ax.set_ylabel("Absolute crude log-risk contrast")
    ax.set_title(cfg["label"], fontweight="bold")
    ax.text(0.03, 0.95, f"Moran I = {moran_i(app['crude_log_risk'], app['adjacency']):.3f}", transform=ax.transAxes, va="top")

    ax = axes[1, column]
    grouped = app["edge_df"].groupby("dissimilarity_quintile", observed=True)["risk_contrast"]
    means = grouped.mean().reindex(["Q1", "Q2", "Q3", "Q4", "Q5"])
    q25 = grouped.quantile(0.25).reindex(means.index)
    q75 = grouped.quantile(0.75).reindex(means.index)
    positions = np.arange(1, 6)
    ax.errorbar(positions, means, yerr=[means - q25, q75 - means], color=color, marker="o", linewidth=2, capsize=3)
    ax.set_xticks(positions, means.index)
    ax.set_xlabel("Covariate-dissimilarity quintile")
    ax.set_ylabel("Mean absolute edge risk contrast")
    ax.text(0.03, 0.95, f"Q5/Q1 = {means.loc['Q5'] / means.loc['Q1']:.2f}", transform=ax.transAxes, va="top")

for ax in axes.flat:
    style_boxed_axes(ax)
save_figure(fig, "application_shared_model_diagnostics.png")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), constrained_layout=True)
for ax, (key, app) in zip(axes, applications.items()):
    cfg = app["config"]
    color = APP_COLORS[key]
    ax.scatter(app["covariate"], app["crude_log_risk"], s=20, alpha=0.45, color=color, edgecolor="none")
    bins = pd.qcut(pd.Series(app["covariate"]), 5, duplicates="drop")
    binned = pd.DataFrame({"x": app["covariate"], "r": app["crude_log_risk"], "bin": bins}).groupby("bin", observed=True).agg(x=("x", "mean"), r=("r", "mean"))
    ax.plot(binned["x"], binned["r"], color="black", marker="o", linewidth=1.8)
    rho = spearmanr(app["covariate"], app["crude_log_risk"]).statistic
    ax.set_title(cfg["label"], fontweight="bold")
    ax.set_xlabel(cfg["covariate_label"]); ax.set_ylabel("Crude log risk")
    ax.text(0.03, 0.95, f"Spearman rho = {rho:.3f}", transform=ax.transAxes, va="top")
    style_boxed_axes(ax)
save_figure(fig, "application_covariate_risk_relationships.png")
plt.show()

## Geographic reference atlases and area crosswalks

Directly printing every area name on a national or metropolitan overview would be unreadable, especially for Glasgow and South Korea. Each atlas therefore combines a colored overview with four geographic zoom regions. Stable labels (`G`, `C`, and `K`) identify areas in the zoom panels and are linked to official names and codes through exported crosswalks. California county names are also printed directly because its zoom panels remain sufficiently sparse.

In [ ]:
ATLAS_REGIONS = ["NW", "NE", "SW", "SE"]
ATLAS_COLORS = {"NW": "#D6E7F0", "NE": "#F6D7B8", "SW": "#D9E8D1", "SE": "#E6D7EA"}


def prepare_atlas_gdf(app):
    projected = get_plot_gdf(app["gdf"]).copy()
    representative = projected.geometry.representative_point()
    projected["_rep_x"] = representative.x
    projected["_rep_y"] = representative.y
    x_mid = float(projected["_rep_x"].median())
    y_mid = float(projected["_rep_y"].median())
    projected["atlas_region"] = np.select(
        [
            (projected["_rep_x"] < x_mid) & (projected["_rep_y"] >= y_mid),
            (projected["_rep_x"] >= x_mid) & (projected["_rep_y"] >= y_mid),
            (projected["_rep_x"] < x_mid) & (projected["_rep_y"] < y_mid),
        ],
        ["NW", "NE", "SW"],
        default="SE",
    )
    return projected


def padded_bounds(gdf, fraction=0.08):
    min_x, min_y, max_x, max_y = gdf.total_bounds
    pad_x = max((max_x - min_x) * fraction, 1.0)
    pad_y = max((max_y - min_y) * fraction, 1.0)
    return min_x - pad_x, min_y - pad_y, max_x + pad_x, max_y + pad_y


def plot_reference_atlas(app):
    key = app["key"]
    cfg = app["config"]
    projected = prepare_atlas_gdf(app)
    app["atlas_gdf"] = projected

    crosswalk_columns = {
        "display_id": projected["display_id"],
        "area_id": projected[cfg["id_col"]].map(canonical_area_id),
        "area_name": projected["display_name"],
        "atlas_region": projected["atlas_region"],
    }
    if key == "south_korea" and "area_name_korean" in projected.columns:
        crosswalk_columns["area_name_korean"] = projected["area_name_korean"].astype(str)
    crosswalk = pd.DataFrame(crosswalk_columns).sort_values("display_id")
    crosswalk.to_csv(OUTPUT_DIR / f"{key}_area_reference_crosswalk.csv", index=False, encoding="utf-8-sig")

    fig = plt.figure(figsize=(18, 11), constrained_layout=True)
    grid = fig.add_gridspec(2, 3, width_ratios=[1.25, 1.0, 1.0])
    overview_ax = fig.add_subplot(grid[:, 0])
    zoom_axes = [fig.add_subplot(grid[row, column]) for row in range(2) for column in (1, 2)]

    projected.plot(
        ax=overview_ax, color=projected["atlas_region"].map(ATLAS_COLORS),
        edgecolor="0.45", linewidth=0.35,
    )
    overview_ax.set_title(f"{cfg['label']}: area-reference overview", fontsize=14, fontweight="bold")
    overview_ax.axis("off")
    overview_ax.legend(
        handles=[Patch(facecolor=ATLAS_COLORS[region], edgecolor="0.4", label=region) for region in ATLAS_REGIONS],
        title="Zoom region", loc="lower left", frameon=False,
    )

    for region, ax in zip(ATLAS_REGIONS, zoom_axes):
        subset = projected.loc[projected["atlas_region"] == region].copy()
        projected.plot(ax=ax, color="#F3F2EE", edgecolor="#D8D6CF", linewidth=0.25)
        subset.plot(ax=ax, color=ATLAS_COLORS[region], edgecolor="0.35", linewidth=0.45)
        min_x, min_y, max_x, max_y = padded_bounds(subset, fraction=0.10)
        ax.set_xlim(min_x, max_x); ax.set_ylim(min_y, max_y)
        direct_names = key == "california"
        font_size = 5.8 if direct_names else (5.2 if key == "glasgow" else 4.4)
        for _, area in subset.iterrows():
            label = area["display_id"]
            if direct_names:
                label += "\n" + textwrap.fill(str(area["display_name"]), width=12)
            ax.text(
                area["_rep_x"], area["_rep_y"], label, ha="center", va="center", fontsize=font_size,
                color="#172027", bbox=dict(facecolor="white", edgecolor="none", alpha=0.55, pad=0.25),
            )
        ax.set_title(f"{region} zoom ({len(subset)} areas)", fontweight="bold")
        ax.axis("off")

    fig.suptitle(
        f"Geographic reference atlas: {cfg['label']}", fontsize=17, fontweight="bold"
    )
    fig.text(0.5, 0.005, f"Area labels correspond to {key}_area_reference_crosswalk.csv.", ha="center", fontsize=10)
    save_figure(fig, f"{key}_area_reference_atlas.png")
    plt.show()
    return crosswalk


area_crosswalks = {key: plot_reference_atlas(app) for key, app in applications.items()}

## Named boundary-neighborhood atlases

The full-area reference atlases allow every spatial unit to be located. The following zoom atlases serve a different purpose: they identify the strongest ABI-DAGAR boundary edges in geographic context and report the adjacent area names, posterior boundary probability, covariate contrast, and crude standardized risks. Selection is based only on the ABI boundary probability; the `CARBayes` result directory is used because it contains the manuscript ABI edge outputs and geographic edge identifiers.

In [ ]:
ABI_EDGE_RESULTS_DIR = REAL_DATA_DIR / "results_ABI_vs_CARBayes"


def select_diverse_top_edges(edge_results, number=6):
    ranked = edge_results.sort_values(
        ["boundary_prob_abi", "edge_dissimilarity"], ascending=[False, False]
    ).reset_index(drop=True)
    selected_indices = []
    node_uses = {}
    for idx, edge in ranked.iterrows():
        from_id = canonical_area_id(edge["from_id"])
        to_id = canonical_area_id(edge["to_id"])
        if node_uses.get(from_id, 0) >= 2 or node_uses.get(to_id, 0) >= 2:
            continue
        selected_indices.append(idx)
        node_uses[from_id] = node_uses.get(from_id, 0) + 1
        node_uses[to_id] = node_uses.get(to_id, 0) + 1
        if len(selected_indices) == number:
            break
    if len(selected_indices) < number:
        selected_indices.extend(idx for idx in ranked.index if idx not in selected_indices)
    return ranked.loc[selected_indices[:number]].reset_index(drop=True)


def _linework_only(geometry):
    if geometry is None or geometry.is_empty:
        return None
    if geometry.geom_type in {"LineString", "MultiLineString"}:
        return geometry
    parts = [_linework_only(part) for part in getattr(geometry, "geoms", [])]
    parts = [part for part in parts if part is not None and not part.is_empty]
    return unary_union(parts) if parts else None


def shared_boundary_line(from_geom, to_geom, tolerance=0.5):
    exact = _linework_only(from_geom.boundary.intersection(to_geom.boundary))
    if exact is not None and exact.length > 0:
        return exact

    # Atlas geometries are in UTM; this repairs sub-meter overlay slivers.
    near = from_geom.boundary.intersection(to_geom.boundary.buffer(tolerance))
    return _linework_only(near)


def plot_boundary_atlas(app, number=6):
    key = app["key"]
    cfg = app["config"]
    result_path = ABI_EDGE_RESULTS_DIR / key / "edge_comparison.csv"
    if not result_path.exists():
        print(f"Skipping {key} boundary atlas; missing {result_path}")
        return pd.DataFrame()

    edge_results = pd.read_csv(result_path)
    selected = select_diverse_top_edges(edge_results, number=number)
    projected = app.get("atlas_gdf", prepare_atlas_gdf(app))
    id_to_index = {canonical_area_id(area_id): idx for idx, area_id in enumerate(app["adjacency_ids"])}
    records = []

    fig, axes = plt.subplots(2, 3, figsize=(18, 11), constrained_layout=True)
    for rank, (ax, edge) in enumerate(zip(axes.flat, selected.itertuples(index=False)), start=1):
        from_key = canonical_area_id(edge.from_id)
        to_key = canonical_area_id(edge.to_id)
        from_idx = id_to_index[from_key]
        to_idx = id_to_index[to_key]
        from_area = projected.iloc[from_idx]
        to_area = projected.iloc[to_idx]
        from_geom = from_area.geometry
        to_geom = to_area.geometry
        pair_gdf = projected.iloc[[from_idx, to_idx]]
        min_x, min_y, max_x, max_y = padded_bounds(pair_gdf, fraction=0.75)
        view_box = box(min_x, min_y, max_x, max_y)
        context = projected.loc[projected.geometry.intersects(view_box)]
        context.plot(ax=ax, color="#EFEDE7", edgecolor="#BEBAB0", linewidth=0.45)
        projected.iloc[[from_idx]].plot(ax=ax, color="#3B82A0", edgecolor="white", linewidth=1.1)
        projected.iloc[[to_idx]].plot(ax=ax, color="#D06B3C", edgecolor="white", linewidth=1.1)
        shared_border = shared_boundary_line(from_geom, to_geom)
        if shared_border is not None and not shared_border.is_empty:
            gpd.GeoSeries([shared_border], crs=projected.crs).plot(ax=ax, color="#8E1B1B", linewidth=3.2)

        for area, text_color in [(from_area, "#17495D"), (to_area, "#7D311B")]:
            ax.text(
                area["_rep_x"], area["_rep_y"], textwrap.fill(str(area["display_name"]), width=18),
                ha="center", va="center", fontsize=8, fontweight="bold", color=text_color,
                bbox=dict(facecolor="white", edgecolor=text_color, alpha=0.88, boxstyle="round,pad=0.25"),
            )
        ax.set_xlim(min_x, max_x); ax.set_ylim(min_y, max_y); ax.axis("off")

        from_name = str(from_area["display_name"])
        to_name = str(to_area["display_name"])
        covariate_difference = abs(app["covariate"][from_idx] - app["covariate"][to_idx])
        pair_title = textwrap.fill(f"{rank}. {from_name} - {to_name}", width=48)
        ax.set_title(
            pair_title + f"\nP(boundary)={edge.boundary_prob_abi:.3f}; $|\\Delta x|$={covariate_difference:.2f}; "
            + f"{cfg['ratio_label']}={app['ratio'][from_idx]:.3f} vs {app['ratio'][to_idx]:.3f}",
            fontsize=10, fontweight="bold",
        )
        records.append({
            "rank": rank, "from_id": from_key, "from_name": from_name, "to_id": to_key, "to_name": to_name,
            "boundary_probability_abi": edge.boundary_prob_abi, "standardized_edge_dissimilarity": edge.edge_dissimilarity,
            "raw_covariate_difference": covariate_difference,
            f"from_{cfg['ratio_label'].lower()}": app["ratio"][from_idx],
            f"to_{cfg['ratio_label'].lower()}": app["ratio"][to_idx],
        })

    fig.suptitle(f"Selected high-probability ABI-DAGAR boundary neighborhoods: {cfg['label']}", fontsize=17, fontweight="bold")
    save_figure(fig, f"{key}_named_boundary_atlas.png")
    plt.show()
    boundary_crosswalk = pd.DataFrame(records)
    boundary_crosswalk.to_csv(OUTPUT_DIR / f"{key}_named_boundary_crosswalk.csv", index=False, encoding="utf-8-sig")
    return boundary_crosswalk


named_boundary_crosswalks = {key: plot_boundary_atlas(app) for key, app in applications.items()}


def plot_numbered_boundary_overviews(applications, crosswalks):
    label_offsets = {
        "glasgow": [(-10, 18), (20, -10), (-22, -8), (18, 14), (-18, 14), (0, -20)],
        "california": [(0, 18), (20, -12), (-22, -12), (20, 15), (-20, 15), (0, -20)],
        "south_korea": [(0, 30), (38, 8), (12, -34), (-42, -8), (-34, 28), (40, -28)],
    }
    fig, axes = plt.subplots(1, 3, figsize=(16, 5.8), constrained_layout=True)
    for ax, (key, app) in zip(axes, applications.items()):
        cfg = app["config"]
        projected = app.get("atlas_gdf", prepare_atlas_gdf(app))
        id_to_index = {canonical_area_id(area_id): idx for idx, area_id in enumerate(app["adjacency_ids"])}
        projected.plot(ax=ax, color="#F1EFE9", edgecolor="#A9A59B", linewidth=0.35)

        for edge in crosswalks[key].itertuples(index=False):
            from_idx = id_to_index[canonical_area_id(edge.from_id)]
            to_idx = id_to_index[canonical_area_id(edge.to_id)]
            shared_border = shared_boundary_line(
                projected.iloc[from_idx].geometry, projected.iloc[to_idx].geometry
            )
            if shared_border is None or shared_border.is_empty:
                continue
            gpd.GeoSeries([shared_border], crs=projected.crs).plot(
                ax=ax, color="#9F1D20", linewidth=3.2, zorder=3
            )
            point = shared_border.centroid
            offset = label_offsets[key][int(edge.rank) - 1]
            ax.annotate(
                str(int(edge.rank)), xy=(point.x, point.y), xytext=offset, textcoords="offset points",
                ha="center", va="center", fontsize=9, fontweight="bold", color="#7A1518",
                bbox=dict(boxstyle="circle,pad=0.28", facecolor="white", edgecolor="#9F1D20", linewidth=1.2),
                arrowprops=dict(arrowstyle="-", color="#9F1D20", linewidth=0.9), zorder=4,
            )

        ax.set_title(cfg["label"], fontsize=13, fontweight="bold")
        ax.axis("off")

    fig.text(0.5, 0.01, "Numbers correspond to the application-specific named-boundary crosswalks.", ha="center", fontsize=10)
    save_figure(fig, "application_numbered_boundary_overviews.png")
    plt.show()


plot_numbered_boundary_overviews(applications, named_boundary_crosswalks)

print("\nExploratory outputs written to:", OUTPUT_DIR)
for output_path in sorted(OUTPUT_DIR.glob("*")):
    if output_path.suffix.lower() in {".png", ".pdf", ".csv"}:
        print(" -", output_path.name)